In [ ]:
import json
import pandas as pd
import numpy as np

import cv2 
from typing import TypedDict

import matplotlib.pyplot as plt
import seaborn as sns


import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *
from src.data.plots import image_category

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

STATS_DIR = ANALYSIS_DIR / "image_properties"

In [ ]:
STATS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(CSV_DIR / "data/metadata.csv")
df["boxes"] = df["boxes"].apply(json.loads)

In [ ]:
class ImageProperties(TypedDict):
    path: str
    split: str
    image_category: str
    
    width: int
    height: int
    area: int

    coverage : float
    
    brightness: float
    sharpness: float
    contrast: float


In [ ]:
def create_masks(boxes, width, height):
    masks = {
        "all": np.zeros((height, width), dtype=bool),
        "smoke": np.zeros((height, width), dtype=bool),
        "fire": np.zeros((height, width), dtype=bool)
    }

    for box in boxes:
        xc, yc, w, h = box["coordinates"]

        # coordinates are already cleaned
        x1 = int(np.floor((xc - w / 2) * width))
        y1 = int(np.floor((yc - h / 2) * height))
        x2 = int(np.ceil((xc + w / 2) * width))
        y2 = int(np.ceil(yc + h / 2) * height)

        assert 0 <= x1 < x2 <= width
        assert 0 <= y1 < y2 <= height

        category = "smoke" if box["category"] == 0 else "fire"

        masks[category][y1:y2, x1:x2] = True
        masks["all"][y1:y2, x1:x2] = True
    return masks

In [ ]:
df["category"] = df["boxes"].apply(image_category)

REGIONS = ["global", "background", "all", "fire", "smoke"]
PROPS = ["brightness", "contrast", "sharpness"]

In [ ]:
stats = {}

# background is exluded for now as theres a couple calculations in verifying that
expected = {
    "global": len(df),
    "all": df["boxes"].apply(bool).sum(),
    "fire": df["boxes"].apply(
        lambda boxes: any(box["category"] == 1 for box in boxes)
    ).sum(),
    "smoke": df["boxes"].apply(
        lambda boxes: any(box["category"] == 0 for box in boxes)
    ).sum()
}

valid = True

for region in REGIONS:
    path = STATS_DIR / f"{region}.csv"

    if not path.exists():
        print(f"Path for {region} does not exist")
        valid = False
        break

    stats[region] = pd.read_csv(path)

    # background is excluded for now as each image needs to be processed beforehand to see if the background is valid
    if region == "background":
        continue

    if len(stats[region]) != expected[region]:
        print(f"Unexpected length for {region}")
        valid = False
        break

# if every other region type was valid create background regions
if valid:
    fully_covered = set(
        stats["all"].loc[
            stats["all"]["coverage"] >= 1.0, "path"
        ]
    )

    expected_background = set(df["path"]) - fully_covered

    background = stats["background"]

    if (
        len(background) != len(expected_background)
        or set(background["path"]) != expected_background
    ):
        print("Unexpected background records")
        valid = False
        
if not valid:
    stats = {region: [] for region in REGIONS}

    for row in df.itertuples():
        img = cv2.imread(str(DATASET_DIR / row.path))

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        laplacian = cv2.Laplacian(gray, cv2.CV_64F)

        h, w = gray.shape
        
        masks = create_masks(row.boxes, w, h)

        masks["global"] = np.ones((h, w), dtype=bool)
        masks["background"] = ~masks["all"]

        for region, mask in masks.items():
            if not mask.any():
                continue

            properties = ImageProperties(
                path=row.path,
                split=row.split,
                image_category=row.category,

                width=w,
                height=h,
                area=w * h,

                coverage=mask.mean(),

                brightness=gray[mask].mean(),
                sharpness=laplacian[mask].var(),
                contrast=gray[mask].std(),
            )

            stats[region].append(properties)

    for region in REGIONS:
        stats[region] = pd.DataFrame(stats[region])
        stats[region].to_csv(STATS_DIR / f"{region}.csv", index=False)

stats["global"]

In [ ]:
stats["all"]

In [ ]:
stats["fire"]

In [ ]:
comparisons = {}

for region in ("all", "fire", "smoke"):

    # Joins images with the properties of images one -> one
    comparison = stats[region][
        ["path", "split", "image_category", "coverage", *PROPS]
    ].merge(
        stats["background"][["path", *PROPS]],
        on="path",
        suffixes=(f"_{region}", "_background"),
        validate="one_to_one"
    )

    for prop in PROPS:
        comparison[f"{prop}_delta"] = (
            comparison[f"{prop}_{region}"]
            - comparison[f"{prop}_background"]
        )

        comparison[f"{prop}_abs_delta"] = comparison[f"{prop}_delta"].abs()

    comparisons[region] = comparison
    comparison.to_csv(STATS_DIR / f"{region}_compare_background.csv", index=False)

In [ ]:
comparisons["smoke"]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats["global"][prop][stats["global"]["image_category"] == "neither"],
            stats["global"][prop][stats["global"]["image_category"] == "both"],
            stats["global"][prop][stats["global"]["image_category"] == "smoke_only"],
            stats["global"][prop][stats["global"]["image_category"] == "fire_only"],
        ],
        tick_labels=["Neither", "Both", "Smoke", "Fire"]
    )
    ax.set_title(prop)

plt.savefig(PLOT_DIR / "image_prop_box_plots.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats["global"][prop][stats["global"]["split"] == split]          
            for split in ["train", "val", "test"]
        ],
        tick_labels=["Train", "Val", "Test"],
        showfliers=False
    )

    ax.set_title(prop.capitalize())
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(PLOT_DIR / "image_props_by_split.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

regions = ["global", "background", "smoke", "fire"]

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [stats[region][prop].dropna() for region in regions],
        tick_labels=["Global", "Background", "Smoke", "Fire"],
        showfliers=False
    )

    ax.set_title(prop.capitalize())
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(PLOT_DIR / "regional_image_properties.svg")
plt.show()

In [ ]:
class_data = {
    "fire": comparisons["fire"].loc[
        comparisons["fire"]["image_category"] == "fire_only"
    ].copy(),

    "smoke": comparisons["smoke"].loc[
        comparisons["smoke"]["image_category"] == "smoke_only"
    ].copy()
}

In [ ]:
for region, data in class_data.items():
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    for ax, prop in zip(axes, PROPS):
        ax.boxplot(
            [data[f"{prop}_background"], data[f"{prop}_{region}"]],
            tick_labels=["Background", region.capitalize()],
            showfliers=False
        )

        ax.set_title(prop.capitalize())
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    fig.suptitle(f"{region.capitalize()}-Only Images")
    plt.tight_layout()
    fig.savefig(PLOT_DIR / f"{region}_only_properties.svg")
    plt.show()

In [ ]:
props = ["brightness", "contrast", "sharpness"]

for region, data in class_data.items():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for j, method in enumerate(["pearson", "spearman"]):
        corr = data.corr(method=method, numeric_only=True)

        # Regional properties vs background properties
        sns.heatmap(
            corr.loc[
                [f"{prop}_{region}" for prop in props],
                [f"{prop}_background" for prop in props]
            ],
            ax=axes[0, j],
            annot=True,
            fmt=".2f",
            cmap="viridis",
            vmin=-1,
            vmax=1,
            center=0,
            cbar=False
        )
        axes[0, j].tick_params(axis="x", labelrotation=0)
        axes[0, j].tick_params(axis="y", labelrotation=0)

        axes[0, j].set_title(f"{method.capitalize()} - Region vs Background")

        # Coverage vs differences
        sns.heatmap(
            corr.loc[
                ["coverage"],
                [f"{prop}_{suffix}"
                 for prop in props
                 for suffix in ("delta", "abs_delta")]
            ],
            ax=axes[1, j],
            annot=True,
            fmt=".2f",
            cmap="viridis",
            vmin=-1,
            vmax=1,
            center=0,
            cbar=False
        )

        axes[1, j].set_title(f"{method.capitalize()} - Coverage vs Differences")

    fig.suptitle(f"{region.capitalize()}-Only Correlations", fontsize=16)

    plt.tight_layout()
    fig.savefig(PLOT_DIR / f"{region}_only_correlations.svg")
    plt.show()

In [ ]:
for region, data in class_data.items():
    print(f"\n{region.capitalize()}-only images")

    for prop in PROPS:
        delta = data[f"{prop}_delta"]

        print(
            f"{prop.capitalize()}: "
            f"{(delta > 0).mean():.1%} higher in {region} regions, "
            f"median delta = {delta.median():.2f}"
        )

In [ ]:
print("Coverage vs background brightness")
for region, data in class_data.items():
    pearson = data["coverage"].corr(data["brightness_background"], method="pearson")
    spearman = data["coverage"].corr(data["brightness_background"], method="spearman")

    print(f"{region.capitalize()}: Pearson = {pearson:.3f}, Spearman = {spearman:.3f}")